# TransJakarta Bus Allocation — 1D LSTM Forecasting

**Input:** `bus_besar_all.csv` dan `bus_kecil_all.csv`  
**Model:** 1D LSTM — sequence 7 hari ke belakang → prediksi 7 hari ke depan  
**Output:** Berapa bus (1 atau 2) per corridor per jam, untuk 7 hari ke depan  
**Diproses terpisah:** Bus Besar (priority seat) & Bus Kecil (kapasitas)


In [ ]:
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn -q


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 50)
tf.random.set_seed(42)
np.random.seed(42)


## Config

In [ ]:
BIG_BUS_CSV   = "bus_besar_all.csv"
SMALL_BUS_CSV = "bus_kecil_all.csv"

PRIORITY_THRESH = 6    # priority seat threshold → senior per bus besar
SMALL_CAPACITY  = 15   # kapasitas bus kecil
SENIOR_CUTOFF   = 1980 # born <= ini = senior (Baby Boomer + Gen X)

LOOKBACK_DAYS   = 7    # panjang sequence input LSTM (7 hari ke belakang)
FORECAST_DAYS   = 7    # prediksi 7 hari ke depan

EPOCHS          = 50
BATCH_SIZE      = 32
PATIENCE        = 10   # early stopping


## Definisi Koridor per Tipe Bus

In [ ]:
big_bus = [
    '1','2','3','4','5','6','7','8','9','10','11','12','13','14',
    '1A','1B','1C','1E','1F','1H','1K','1M','1N','1P','1Q','1R','1T',
    '2A','2B','2E','2F','2H','2P','2Q',
    '3A','3B','3C','3E','3F','3H',
    '4B','4C','4D','4E','4F',
    '5B','5C','5D','5F','5M','5N',
    '6A','6B','6C','6D','6H','6M','6N','6P','6Q','6T','6U','6V',
    '7A','7B','7C','7D','7E','7F','7P','7Q',
    '8A','8C','8D','8E','8K','8M',
    '9A','9C','9D','9E','9F','9H','9N',
    '10A','10B','10D','10H','10K',
    '11B','11C','11D','11K','11M','11N','11P','11Q',
    '12A','12B','12C','12F','12H','12P',
    '13B','13C','13D',
]
big_bus_regional = [
    'B11','B13','B14','B21','D11','D21','D31','D32',
    'S11','S12','S21','S22','S31','T11','T21',
    'R1A','BW9','JIS3','L13E',
]
mikrotrans = [
    'JAK.01','JAK.02','JAK.03','JAK.04','JAK.05','JAK.06','JAK.07','JAK.08',
    'JAK.10','JAK.11','JAK.12','JAK.13','JAK.14','JAK.15','JAK.16','JAK.17',
    'JAK.18','JAK.19','JAK.20','JAK.21','JAK.22','JAK.23','JAK.24','JAK.25',
    'JAK.26','JAK.27','JAK.28','JAK.29','JAK.30','JAK.31','JAK.32','JAK.33',
    'JAK.34','JAK.35','JAK.36','JAK.37','JAK.38','JAK.39','JAK.40','JAK.41',
    'JAK.42','JAK.43B','JAK.43C','JAK.44','JAK.45','JAK.46','JAK.47',
    'JAK.48A','JAK.48B','JAK.49','JAK.50','JAK.51','JAK.52','JAK.53',
    'JAK.54','JAK.56','JAK.58','JAK.59','JAK.60','JAK.61','JAK.64',
    'JAK.71','JAK.72','JAK.73','JAK.74','JAK.75','JAK.77','JAK.80',
    'JAK.84','JAK.85','JAK.86','JAK.88','JAK.99',
    'JAK.106','JAK.110A','JAK.112','JAK.113','JAK.115','JAK.117','JAK.118','JAK.120',
]
minitrans = ['M1','M1H','M2','M3','M4','M5','M6','M7','M7B','M8','M9','M10','M11','M12','M13']

ALL_BIG   = set(big_bus + big_bus_regional)
ALL_SMALL = set(mikrotrans + minitrans)


## Step 1: Load & Compute Hourly Peaks

Dari CSV transaksi:
- Drop lat/lon
- Simulasi tap-in (+1) / tap-out (−1) per corridor per jam
- Hitung `peak_active` dan `peak_senior_active`


In [ ]:
def load_and_aggregate(csv_path: str, bus_type: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)

    # Drop lat/lon
    drop_cols = [c for c in df.columns if "Lat" in c or "Lon" in c]
    df = df.drop(columns=drop_cols, errors="ignore")

    df["tapInTime"]        = pd.to_datetime(df["tapInTime"],  errors="coerce")
    df["tapOutTime"]       = pd.to_datetime(df["tapOutTime"], errors="coerce")
    df["payCardBirthDate"] = pd.to_numeric(df["payCardBirthDate"], errors="coerce")

    df = df.dropna(subset=["tapInTime", "tapOutTime", "corridorID"])
    df = df[df["tapOutTime"] >= df["tapInTime"]].copy()

    if bus_type == "big":
        df = df[df["corridorID"].isin(ALL_BIG)].copy()
    else:
        df = df[df["corridorID"].isin(ALL_SMALL)].copy()

    df["date"]      = df["tapInTime"].dt.date
    df["hour"]      = df["tapInTime"].dt.hour
    df["is_senior"] = (df["payCardBirthDate"] <= SENIOR_CUTOFF).astype(int)

    print(f"[{bus_type}] Rows: {len(df):,} | Date range: {df['date'].min()} → {df['date'].max()} | Corridors: {df['corridorID'].nunique()}")

    records = []
    for (corridor, date, hour), grp in df.groupby(["corridorID", "date", "hour"]):
        ev_in             = grp[["tapInTime", "is_senior"]].rename(columns={"tapInTime": "event_time"}).copy()
        ev_in["d_all"]    =  1
        ev_in["d_senior"] =  ev_in["is_senior"]

        same_hour_out     = grp[grp["tapOutTime"].dt.hour == hour][["tapOutTime", "is_senior"]].rename(columns={"tapOutTime": "event_time"}).copy()
        same_hour_out["d_all"]    = -1
        same_hour_out["d_senior"] = -same_hour_out["is_senior"]

        events = pd.concat(
            [ev_in[["event_time","d_all","d_senior"]],
             same_hour_out[["event_time","d_all","d_senior"]]],
            ignore_index=True
        ).sort_values("event_time")

        events["active_all"]    = events["d_all"].cumsum().clip(lower=0)
        events["active_senior"] = events["d_senior"].cumsum().clip(lower=0)

        records.append({
            "corridorID"         : corridor,
            "date"               : pd.Timestamp(date),
            "hour"               : hour,
            "peak_active"        : int(events["active_all"].max()),
            "peak_senior_active" : int(events["active_senior"].max()),
            "total_tapin"        : len(grp),
        })

    return pd.DataFrame(records).sort_values(["corridorID","date","hour"]).reset_index(drop=True)


## Step 2: Derive Label — `buses_needed` (1 atau 2)

| Bus | Kondisi | Label |
|-----|---------|-------|
| Besar | `peak_senior_active > 6` | 2 |
| Besar | `peak_senior_active ≤ 6` | 1 |
| Kecil | `peak_active > 15` | 2 |
| Kecil | `peak_active ≤ 15` | 1 |


In [ ]:
def derive_labels(hourly: pd.DataFrame, bus_type: str) -> pd.DataFrame:
    h = hourly.copy()
    if bus_type == "big":
        h["buses_needed"] = (h["peak_senior_active"] > PRIORITY_THRESH).astype(int) + 1
    else:
        h["buses_needed"] = (h["peak_active"] > SMALL_CAPACITY).astype(int) + 1
    h["buses_needed"] = h["buses_needed"].clip(1, 2)
    return h


## Step 3: Build LSTM Sequences

Untuk setiap `(corridorID, hour)` kita punya time series harian.

**Input sequence (X):** 7 hari ke belakang × fitur  
**Target (y):** `buses_needed` pada hari ke-8 s.d. ke-14 (7 hari ke depan)  

Fitur per timestep (per hari):
- `buses_needed` (historis)
- `peak_active`, `peak_senior_active`, `total_tapin`
- `day_of_week`, `is_weekend`

Zero data leak: X selalu berasal dari hari-hari sebelum y.


In [ ]:
FEATURE_COLS = ["buses_needed", "peak_active", "peak_senior_active", "total_tapin",
                "day_of_week", "is_weekend"]

def make_sequences(series_df: pd.DataFrame) -> tuple:
    """
    Dari time series 1 corridor+jam, buat sliding window sequences.
    X shape: (n_samples, LOOKBACK_DAYS, n_features)
    y shape: (n_samples, FORECAST_DAYS)  — buses_needed untuk 7 hari ke depan
    """
    series_df = series_df.sort_values("date").copy()
    series_df["day_of_week"] = series_df["date"].dt.dayofweek
    series_df["is_weekend"]  = (series_df["day_of_week"] >= 5).astype(int)

    vals  = series_df[FEATURE_COLS].values.astype(np.float32)
    label = series_df["buses_needed"].values.astype(np.float32)

    n     = len(vals)
    X_list, y_list = [], []

    # Setiap window: input = [t-7 .. t-1], target = [t .. t+6]
    for i in range(LOOKBACK_DAYS, n - FORECAST_DAYS + 1):
        X_list.append(vals[i - LOOKBACK_DAYS : i])          # (7, n_features)
        y_list.append(label[i : i + FORECAST_DAYS])         # (7,)

    if not X_list:
        return None, None

    return np.array(X_list), np.array(y_list)


def build_lstm_dataset(hourly: pd.DataFrame):
    """
    Iterasi semua (corridorID, hour), kumpulkan sequences.
    Return X, y, scaler (fit pada train), dan metadata per sequence.
    """
    X_all, y_all, meta_all = [], [], []

    for (corridor, hour), grp in hourly.groupby(["corridorID", "hour"]):
        grp = grp.sort_values("date").reset_index(drop=True)
        if len(grp) < LOOKBACK_DAYS + FORECAST_DAYS:
            continue  # tidak cukup hari

        X, y = make_sequences(grp)
        if X is None:
            continue

        for i in range(len(X)):
            X_all.append(X[i])
            y_all.append(y[i])
            meta_all.append({"corridorID": corridor, "hour": hour})

    return np.array(X_all), np.array(y_all), pd.DataFrame(meta_all)


## Step 4: Scale & Chronological Split

- Scale fitur dengan `MinMaxScaler` — fit **hanya pada train sequences**
- Split: train = semua kecuali window 7+7 hari terakhir, test = sisa
- Tidak ada data leak


In [ ]:
def scale_and_split(X: np.ndarray, y: np.ndarray, meta: pd.DataFrame, hourly: pd.DataFrame):
    """
    Chronological split berdasarkan urutan tanggal terakhir dari tiap sequence.
    Sequences yang window-nya entirely sebelum cutoff = train, sisanya = test.
    """
    all_dates = hourly["date"].sort_values().unique()
    n_dates   = len(all_dates)

    # Tanggal cutoff: kita sisakan LOOKBACK+FORECAST hari terakhir untuk test
    cutoff_idx  = n_dates - LOOKBACK_DAYS - FORECAST_DAYS
    cutoff_date = all_dates[cutoff_idx]

    # Untuk setiap sequence, identifikasi tanggal awal window-nya
    # Urutan sequences di X_all sesuai urutan groupby corridorID+hour lalu sliding,
    # kita pakai meta sebagai penanda, tapi kita perlu tanggal window-nya.
    # Cara paling clean: rebuild tanggal end-of-window per sequence
    date_ends = []
    for (corridor, hour), grp in hourly.groupby(["corridorID", "hour"]):
        grp = grp.sort_values("date").reset_index(drop=True)
        if len(grp) < LOOKBACK_DAYS + FORECAST_DAYS:
            continue
        dates = grp["date"].values
        for i in range(LOOKBACK_DAYS, len(dates) - FORECAST_DAYS + 1):
            date_ends.append(dates[i - 1])   # tanggal akhir input window

    date_ends = np.array(date_ends)

    train_mask = date_ends <= cutoff_date
    test_mask  = ~train_mask

    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    meta_test         = meta[test_mask].reset_index(drop=True)

    print(f"  Train sequences: {len(X_train):,}  |  Test sequences: {len(X_test):,}")

    # Scale fitur — fit pada train only
    n_feats = X_train.shape[2]
    scaler  = MinMaxScaler()
    X_train_2d = X_train.reshape(-1, n_feats)
    X_test_2d  = X_test.reshape(-1, n_feats)

    scaler.fit(X_train_2d)
    X_train = scaler.transform(X_train_2d).reshape(X_train.shape)
    X_test  = scaler.transform(X_test_2d).reshape(X_test.shape)

    return X_train, y_train, X_test, y_test, meta_test, scaler, cutoff_date


## Step 5: Build & Train 1D LSTM

Arsitektur:
```
Input  (7, n_features)
  → LSTM(64, return_sequences=True)
  → Dropout(0.2)
  → LSTM(32)
  → Dropout(0.2)
  → Dense(7, activation='sigmoid')   # output 7 hari, nilai 0–1
Output → round → 1 atau 2 bus
```

Output LSTM adalah nilai kontinu 0–1 (sigmoid), lalu di-threshold:  
`buses_needed_pred = 1 + round(output)` → 1 atau 2


In [ ]:
def build_lstm(n_features: int) -> tf.keras.Model:
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOKBACK_DAYS, n_features)),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(FORECAST_DAYS, activation="sigmoid"),  # 7 output nodes
    ])
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["mae"],
    )
    model.summary()
    return model


def train_lstm(X_train, y_train, X_test, y_test, n_features: int):
    # y harus dinormalisasi ke 0–1 (buses_needed − 1)
    y_tr = (y_train - 1).astype(np.float32)
    y_te = (y_test  - 1).astype(np.float32)

    model = build_lstm(n_features)

    history = model.fit(
        X_train, y_tr,
        validation_data=(X_test, y_te),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)],
        verbose=1,
    )
    return model, history


## Step 6: Evaluasi Model

In [ ]:
def evaluate_model(model, X_test, y_test, history, bus_type: str):
    y_te_norm = (y_test - 1).astype(np.float32)
    y_pred_raw = model.predict(X_test, verbose=0)         # (n, 7), nilai 0–1
    y_pred     = np.round(y_pred_raw) + 1                 # 1 atau 2
    y_actual   = y_test                                   # 1 atau 2

    # Accuracy per hari ke-1 s.d. ke-7
    acc_per_day = [(y_pred[:, d] == y_actual[:, d]).mean() for d in range(FORECAST_DAYS)]

    print(f"\n{'='*50}")
    print(f"[{bus_type.upper()}] Accuracy per hari prediksi:")
    for d, acc in enumerate(acc_per_day):
        print(f"  Hari+{d+1}: {acc:.4f}")
    print(f"  Overall: {(y_pred == y_actual).mean():.4f}")

    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"Bus {'Besar' if bus_type=='big' else 'Kecil'} — Training History", fontsize=13)

    axes[0].plot(history.history["loss"],     label="Train Loss")
    axes[0].plot(history.history["val_loss"], label="Val Loss")
    axes[0].set_title("Loss (Binary Crossentropy)")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].bar([f"H+{d+1}" for d in range(FORECAST_DAYS)], acc_per_day, color="steelblue")
    axes[1].axhline(0.5, color="red", linestyle="--", label="Baseline (0.5)")
    axes[1].set_title("Accuracy per Hari Prediksi")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return y_pred, y_actual


## Step 7: Prediksi 7 Hari ke Depan

Untuk setiap `(corridorID, hour)`, ambil **7 hari terakhir** yang ada di data sebagai input sequence, lalu prediksi 7 hari ke depan.

Output: tabel `corridorID × tanggal_prediksi × jam → buses_needed_pred`


In [ ]:
def forecast_next_7_days(model, hourly: pd.DataFrame, scaler, bus_type: str) -> pd.DataFrame:
    """
    Untuk tiap (corridorID, hour): ambil 7 hari terakhir sebagai input,
    prediksi 7 hari ke depan.
    """
    last_date    = hourly["date"].max()
    future_dates = [last_date + pd.Timedelta(days=d+1) for d in range(FORECAST_DAYS)]
    n_features   = len(FEATURE_COLS)

    records = []

    for (corridor, hour), grp in hourly.groupby(["corridorID", "hour"]):
        grp = grp.sort_values("date").reset_index(drop=True)
        if len(grp) < LOOKBACK_DAYS:
            continue

        grp["day_of_week"] = grp["date"].dt.dayofweek
        grp["is_weekend"]  = (grp["day_of_week"] >= 5).astype(int)

        # Ambil 7 hari terakhir sebagai input window
        window = grp[FEATURE_COLS].values[-LOOKBACK_DAYS:].astype(np.float32)  # (7, n_features)

        # Scale menggunakan scaler yang sudah di-fit pada train
        window_scaled = scaler.transform(window)  # (7, n_features)
        X_input = window_scaled[np.newaxis, :, :]  # (1, 7, n_features)

        # Predict
        pred_raw  = model.predict(X_input, verbose=0)[0]  # (7,)
        pred_bus  = np.round(pred_raw) + 1                # 1 atau 2

        for d_idx, (fut_date, pred_val) in enumerate(zip(future_dates, pred_bus)):
            records.append({
                "corridorID"       : corridor,
                "hour"             : hour,
                "predicted_date"   : fut_date.date(),
                "day_ahead"        : d_idx + 1,
                "day_of_week"      : fut_date.dayofweek(),
                "buses_needed_pred": int(pred_val),
            })

    forecast_df = pd.DataFrame(records).sort_values(
        ["predicted_date", "corridorID", "hour"]
    ).reset_index(drop=True)

    return forecast_df


## Step 8: Visualisasi Prediksi 7 Hari ke Depan

In [ ]:
def plot_forecast_heatmap(forecast_df: pd.DataFrame, bus_type: str, top_n: int = 25):
    """
    Heatmap: corridorID (top-N yang paling sering butuh 2 bus) × predicted_date
    Nilai = avg buses_needed_pred (1.0–2.0, di atas 1.5 = sering butuh 2 bus)
    """
    top_corridors = (
        forecast_df[forecast_df["buses_needed_pred"] == 2]
        .groupby("corridorID")["buses_needed_pred"]
        .count()
        .sort_values(ascending=False)
        .head(top_n)
        .index.tolist()
    )
    if not top_corridors:
        top_corridors = forecast_df["corridorID"].value_counts().head(top_n).index.tolist()

    pivot = (
        forecast_df[forecast_df["corridorID"].isin(top_corridors)]
        .groupby(["corridorID", "predicted_date"])["buses_needed_pred"]
        .mean()
        .unstack()
    )
    pivot.columns = [str(c) for c in pivot.columns]

    plt.figure(figsize=(14, max(6, len(top_corridors) * 0.35)))
    sns.heatmap(
        pivot,
        cmap="YlOrRd", vmin=1, vmax=2,
        annot=True, fmt=".1f",
        linewidths=0.3,
        cbar_kws={"label": "Avg Buses Predicted"},
    )
    label = "Bus Besar" if bus_type == "big" else "Bus Kecil"
    plt.title(f"Prediksi Alokasi — {label}\nTop {top_n} Corridor Paling Butuh 2 Bus (7 Hari ke Depan)")
    plt.xlabel("Tanggal Prediksi")
    plt.ylabel("Corridor ID")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


def plot_forecast_by_hour(forecast_df: pd.DataFrame, bus_type: str, top_n: int = 10):
    """
    Heatmap: jam × hari ke depan, untuk top-N corridor paling sibuk.
    """
    top_corridors = (
        forecast_df[forecast_df["buses_needed_pred"] == 2]
        .groupby("corridorID")["buses_needed_pred"]
        .count()
        .sort_values(ascending=False)
        .head(top_n)
        .index.tolist()
    )
    if not top_corridors:
        top_corridors = forecast_df["corridorID"].value_counts().head(top_n).index.tolist()

    subset = forecast_df[forecast_df["corridorID"].isin(top_corridors)]

    pivot = (
        subset.groupby(["hour", "day_ahead"])["buses_needed_pred"]
        .mean()
        .unstack()
    )
    pivot.columns = [f"H+{c}" for c in pivot.columns]

    plt.figure(figsize=(12, 8))
    sns.heatmap(
        pivot, cmap="YlOrRd", vmin=1, vmax=2,
        annot=True, fmt=".2f",
        linewidths=0.3,
        cbar_kws={"label": "Avg Buses Predicted"},
    )
    label = "Bus Besar" if bus_type == "big" else "Bus Kecil"
    plt.title(f"Pola Jam × Hari ke Depan — {label}\n(Rata-rata top {top_n} corridor paling sibuk)")
    plt.xlabel("Hari ke Depan")
    plt.ylabel("Jam")
    plt.tight_layout()
    plt.show()


In [ ]:
def display_forecast_summary(forecast_df: pd.DataFrame, bus_type: str):
    label = "Bus Besar" if bus_type == "big" else "Bus Kecil"

    print(f"\n{'='*60}")
    print(f"  PREDIKSI 7 HARI KE DEPAN — {label.upper()}")
    print(f"{'='*60}")
    print(f"  Total slot (corridor × jam × hari): {len(forecast_df):,}")
    need_2 = (forecast_df['buses_needed_pred'] == 2).sum()
    print(f"  Slot yang perlu 2 bus: {need_2:,} ({need_2/len(forecast_df)*100:.1f}%)")

    print(f"\n  Summary per Tanggal:")
    summary_date = (
        forecast_df.groupby("predicted_date")["buses_needed_pred"]
        .agg(
            total_slots = "count",
            need_2_buses = lambda x: (x == 2).sum(),
            pct_need_2   = lambda x: f"{(x == 2).mean()*100:.1f}%"
        )
        .reset_index()
    )
    display(summary_date)

    print(f"\n  Corridor yang SELALU butuh 2 bus (semua jam, semua 7 hari):")
    always_2 = (
        forecast_df.groupby("corridorID")["buses_needed_pred"]
        .min()
    )
    always_2 = always_2[always_2 == 2].index.tolist()
    print(f"  {always_2 if always_2 else 'Tidak ada'}")

    print(f"\n  Full prediction table (sample):")
    display(forecast_df.head(30))


---
## 🚌 BUS BESAR — Full Pipeline

In [ ]:
print("=" * 60)
print("BUS BESAR")
print("=" * 60)

hourly_big = load_and_aggregate(BIG_BUS_CSV, bus_type="big")
hourly_big = derive_labels(hourly_big, bus_type="big")

print("\nLabel distribution:")
print(hourly_big["buses_needed"].value_counts().to_string())


In [ ]:
X_big, y_big, meta_big = build_lstm_dataset(hourly_big)
print(f"X shape: {X_big.shape}  |  y shape: {y_big.shape}")


In [ ]:
X_tr_big, y_tr_big, X_te_big, y_te_big, meta_te_big, scaler_big, cutoff_big =     scale_and_split(X_big, y_big, meta_big, hourly_big)

print(f"Cutoff date (akhir train): {cutoff_big}")


In [ ]:
model_big, history_big = train_lstm(
    X_tr_big, y_tr_big,
    X_te_big, y_te_big,
    n_features=len(FEATURE_COLS)
)


In [ ]:
y_pred_big, y_actual_big = evaluate_model(
    model_big, X_te_big, y_te_big, history_big, bus_type="big"
)


In [ ]:
forecast_big = forecast_next_7_days(model_big, hourly_big, scaler_big, bus_type="big")
display_forecast_summary(forecast_big, bus_type="big")


In [ ]:
plot_forecast_heatmap(forecast_big, bus_type="big", top_n=25)
plot_forecast_by_hour(forecast_big, bus_type="big", top_n=10)


---
## 🚐 BUS KECIL — Full Pipeline

In [ ]:
print("=" * 60)
print("BUS KECIL")
print("=" * 60)

hourly_small = load_and_aggregate(SMALL_BUS_CSV, bus_type="small")
hourly_small = derive_labels(hourly_small, bus_type="small")

print("\nLabel distribution:")
print(hourly_small["buses_needed"].value_counts().to_string())


In [ ]:
X_small, y_small, meta_small = build_lstm_dataset(hourly_small)
print(f"X shape: {X_small.shape}  |  y shape: {y_small.shape}")


In [ ]:
X_tr_sm, y_tr_sm, X_te_sm, y_te_sm, meta_te_sm, scaler_sm, cutoff_sm =     scale_and_split(X_small, y_small, meta_small, hourly_small)

print(f"Cutoff date (akhir train): {cutoff_sm}")


In [ ]:
model_small, history_small = train_lstm(
    X_tr_sm, y_tr_sm,
    X_te_sm, y_te_sm,
    n_features=len(FEATURE_COLS)
)


In [ ]:
y_pred_sm, y_actual_sm = evaluate_model(
    model_small, X_te_sm, y_te_sm, history_small, bus_type="small"
)


In [ ]:
forecast_small = forecast_next_7_days(model_small, hourly_small, scaler_sm, bus_type="small")
display_forecast_summary(forecast_small, bus_type="small")


In [ ]:
plot_forecast_heatmap(forecast_small, bus_type="small", top_n=25)
plot_forecast_by_hour(forecast_small, bus_type="small", top_n=10)
